In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from runner import DualRunner

PG_CONNINFO = (
    f"host=127.0.0.1 "
    f"port={os.getenv("POSTGRES_PORT", 5432)} "
    f"dbname={os.getenv("POSTGRES_DB")} "
    f"user={os.getenv("POSTGRES_USER")} "
    f"password={os.getenv("POSTGRES_PASSWORD")}"
)

runner = DualRunner(
    pg_conninfo=PG_CONNINFO,
    duckdb_path=":memory:"
)

In [3]:
display(runner.run_pg("select version()"))
display(runner.run_dd("select version()"))

,version
0,PostgreSQL 17.7 (Debian 17.7-3.pgdg13+1) on aa...


,"""version""()"
0,v1.4.3


In [4]:
runner.check("""--sql

drop table if exists load_sample;
create table load_sample (
    sample_date date PRIMARY KEY,
    load_value integer
);
insert into load_sample (sample_date, load_value)
values
    ('2018-02-01', 1024),
    ('2018-02-02', 2366),
    ('2018-02-05', 2366),
    ('2018-02-07', 985),
    ('2018-02-08', 780),
    ('2018-02-12', 1000)
;
select * from load_sample;
""")

same


,sample_date,load_value
0,2018-02-01,1024
1,2018-02-02,2366
2,2018-02-05,2366
3,2018-02-07,985
4,2018-02-08,780
5,2018-02-12,1000


In [5]:
runner.check("""--sql
select 
    sample_date as current_date,
    load_value as current_load_value,
    min(sample_date) over (
        order by sample_date
        rows between 1 preceding and 1 preceding
        ) as latest_date,
    min(load_value) over (
        order by sample_date
        rows between 1 preceding and 1 preceding
    ) as latest_load,
    min(sample_date) over (
        order by sample_date
        rows between 2 preceding and 2 preceding
    ) as latest_load_2

from load_sample
""")

same


,current_date,current_load_value,latest_date,latest_load,latest_load_2
0,2018-02-01,1024,None,NaN,None
1,2018-02-02,2366,2018-02-01,1024.0,None
2,2018-02-05,2366,2018-02-02,2366.0,2018-02-01
3,2018-02-07,985,2018-02-05,2366.0,2018-02-02
4,2018-02-08,780,2018-02-07,985.0,2018-02-05
5,2018-02-12,1000,2018-02-08,780.0,2018-02-07


In [6]:
runner.check("""--sql
-- 同じwindow定義がある場合は window 句で定義できる
select 
    sample_date as current_date,
    load_value as current_load_value,
    min(sample_date) over w as latest_date,
    min(load_value) over w as latest_load
from load_sample
window w as (
    order by sample_date
    rows between 1 preceding and 1 preceding
)

""")

same


,current_date,current_load_value,latest_date,latest_load
0,2018-02-01,1024,None,NaN
1,2018-02-02,2366,2018-02-01,1024.0
2,2018-02-05,2366,2018-02-02,2366.0
3,2018-02-07,985,2018-02-05,2366.0
4,2018-02-08,780,2018-02-07,985.0
5,2018-02-12,1000,2018-02-08,780.0


- 行に基づく書き方
    - `rows between n preceding and m preceding` : n行前からm行前
    - `rows between n following nad m following` : n行後からm行後
    - `rows between n preceding nad m following` : n行前からm行後
- 行の値に基づく書きかた
    - `range between interval 'n' day preceding and 'm' day preceding`: n日前からn日前

- ROWS：移動単位を行で設定する
- RANGE：移動単位を列の値で設定する。基準となる列はORDER BY句で指定された列
- n PRECEDING：nだけ前へ（小さいほう）へ移動する。nは正の整数 
- n FOLLOWING：nだけ後へ（大きいほう）へ移動する。nは正の整数 
- UNBOUNDED PRECEDING：無制限にさかのぼるほうへ移動する 
- UNBOUNDED FOLLOWING：無制限に下るほうへ移動する 
- CURRENT ROW：現在行

ミック. 達人に学ぶSQL徹底指南書 第2版 初級者で終わりたくないあなたへ (p. 70). (Function). Kindle Edition. 

In [7]:
runner.check("""--sql

drop table if exists server_load_sample;
create table server_load_sample (
    server varchar(10),
    sample_date date,
    load_val integer
);
insert into server_load_sample (server, sample_date, load_val) values
('A', '2018-02-01', 1024),
('A', '2018-02-02', 2366),
('A', '2018-02-05', 2366),
('A', '2018-02-07', 985),
('A', '2018-02-08', 780),
('A', '2018-02-12', 1000),
('B', '2018-02-01', 54),
('B', '2018-02-02', 39008),
('B', '2018-02-03', 2900),
('B', '2018-02-04', 556),
('B', '2018-02-05', 12600),
('B', '2018-02-06', 7309),
('C', '2018-02-01', 1000),
('C', '2018-02-07', 2000),
('C', '2018-02-16', 500);
select * from server_load_sample;
""")

same


,server,sample_date,load_val
0,A,2018-02-01,1024
1,A,2018-02-02,2366
2,A,2018-02-05,2366
3,A,2018-02-07,985
4,A,2018-02-08,780
5,A,2018-02-12,1000
6,B,2018-02-01,54
7,B,2018-02-02,39008
8,B,2018-02-03,2900
9,B,2018-02-04,556


In [8]:
runner.check("""--sql
select
    server,
    sample_date,
    load_val,
    sum(load_val) over () as all_load, -- これは全部の行を sum する　
    sum(load_val) over (partition by server) as sum_by_server
from server_load_sample
""")

same


,server,sample_date,load_val,all_load,sum_by_server
0,A,2018-02-01,1024,74448,8521
1,A,2018-02-02,2366,74448,8521
2,A,2018-02-05,2366,74448,8521
3,A,2018-02-07,985,74448,8521
4,A,2018-02-08,780,74448,8521
5,A,2018-02-12,1000,74448,8521
6,B,2018-02-01,54,74448,62427
7,B,2018-02-02,39008,74448,62427
8,B,2018-02-03,2900,74448,62427
9,B,2018-02-04,556,74448,62427


## 自己結合







In [9]:
runner.check("""--sql

-- 自己結合
drop table if exists fruits;
create table fruits (
    name text
);
insert into fruits (name) values
('apple'), ('mikan'), ('banana');
select * from fruits;

""")

same


,name
0,apple
1,mikan
2,banana


In [ ]:
runner.check("""--sql

-- 単純な順序対
-- 直積を作る
select 
    f1.name as name_1,
    f2.name as name_2
from fruits as f1
cross join fruits as f2 -- cross join は結合条件が存在しない

""")

same


,name_1,name_2
0,apple,apple
1,apple,mikan
2,apple,banana
3,mikan,apple
4,mikan,mikan
5,mikan,banana
6,banana,apple
7,banana,mikan
8,banana,banana


In [12]:
runner.check("""--sql
-- 同じ組み合わせを排除
-- 順序対。順序が関係する組み合わせ。apple-mikan と mikan-apple を別物として扱っている
-- 順序を考慮する
select
    f1.name as name_1,
    f2.name as name_2
from fruits as f1
inner join fruits as f2
    on f1.name != f2.name

""")

same


,name_1,name_2
0,apple,mikan
1,apple,banana
2,mikan,apple
3,mikan,banana
4,banana,apple
5,banana,mikan


In [13]:
runner.check("""--sql
-- 順序を考慮しない組み合わせ = Combination を　得る
-- 非順序対
select
    f1.name as name_1,
    f2.name as name_2
from fruits as f1
inner join fruits as f2
    on f1.name > f2.name
    -- on f1.name < f2.name
-- 不等号の向きはどちらでも問題ない。 apple-mikan と mikan-apple のどちらを組み合わせとして採用するかが変わるだけ

""")

same


,name_1,name_2
0,mikan,apple
1,mikan,banana
2,banana,apple


In [18]:
runner.check("""--sql
-- 非順序対の、3つ以上の組み合わせの求め方は、上記にjoinをつなげていけば良い
select
    f1.name as name_1,
    f2.name as name_2,
    f3.name as name_3
from fruits as f1
inner join fruits as f2
    on f1.name > f2.name
inner join fruits as f3
    on f2.name > f3.name

""")

same


,name_1,name_2,name_3
0,mikan,banana,apple


In [19]:
runner.check("""--sql
-- 部分的に不一致なキーの検索
drop table if exists addresses;
create table addresses (
    name varchar(100),
    family_id integer,
    address varchar(200)
);
insert into addresses (name, family_id, address) values
('前田義明', 100, '東京都港区虎ノ門 3-2-29'),
('前田由美', 100, '東京都港区虎ノ門 3-2-92'),
('加藤茶', 200, '東京都新宿区西新宿 2-8-1'),
('加藤勝', 200, '東京都新宿区西新宿 2-8-1'),
('ホームズ', 300, 'ベーカー街 221B'),
('ワトソン', 400, 'ベーカー街 221B');
select * from addresses;
""")

same


,name,family_id,address
0,前田義明,100,東京都港区虎ノ門 3-2-29
1,前田由美,100,東京都港区虎ノ門 3-2-92
2,加藤茶,200,東京都新宿区西新宿 2-8-1
3,加藤勝,200,東京都新宿区西新宿 2-8-1
4,ホームズ,300,ベーカー街 221B
5,ワトソン,400,ベーカー街 221B


In [22]:
runner.check("""--sql
-- 家族 id が一緒なら同じ住所にする
-- このルールに則っていない行を検索する

select
    a1.name,
    a1.address
from addresses as a1
inner join addresses as a2
    on a1.family_id = a2.family_id
    and a1.address != a2.address

""")


same


,name,address
0,前田義明,東京都港区虎ノ門 3-2-29
1,前田由美,東京都港区虎ノ門 3-2-92


In [ ]:
runner.check("""--sql
-- 自己結合と人うち結合の組み合わせの他の例
drop table if exists fruits2;
create table fruits2 (
    name varchar(100),
    price integer
);
insert into fruits2 (name, price) values
('りんご', 50),
('みかん', 100),
('ぶどう', 50),
('スイカ', 80),
('レモン', 30),
('いちご', 100),
('バナナ', 100);
select * from fruits2;

""")

same


,name,price
0,りんご,50
1,みかん,100
2,ぶどう,50
3,スイカ,80
4,レモン,30
5,いちご,100
6,バナナ,100


In [ ]:
%%sql
-- 値段が同じ商品の組み合わせを得る

select distinct
    f1.name,
    f1.price
from fruits2 as f1
inner join fruits2 as f2
    on f1.name != f2.name
    and f1.price = f2.price
order by price

In [ ]:
%%sql
-- ランキング
select
    name,
    price,
    rank() over (order by price desc) as rank, -- 順位が飛び飛び
    dense_rank() over (order by price desc) as dense_rank, -- 順位を詰める
    row_number() over (order by price desc) as row_number,-- 同順位があっても順位付け
    percent_rank() over (order by price desc) as percent_rank,
    ntile(4) over (order by price desc) as quartile, -- n tile に分ける 同じ price の中でも異なる？
    ceil(rnk * 4.0 / (max(rnk) over ())::numeric) as quartile_by_dense_rank -- 同順位を考慮に入れるなら、手動で計算する　
    -- ceil(rnk * 4 / max_rnk) を計算する
from (
    select 
        *,
        dense_rank() over (order by price desc) as rnk
    from fruits2
) as t

## null値について

- 3値論理
- true, false, unknownの3つ
- unknown は null が含まれる条件式を評価したときに出てくる
    - ちなみに null = null も unkwnon を返す
    - null を正しく評価できるのは null is null とか null is not null だけ
    - null に比較述語を適用した結果が常に unknown になってしまうため
- null には Unknown と Not Applicable の2つの概念がある

- null の厄介な性質
    - 「ジョンは20歳か、20歳でないかのどちらかである」というような排中律が成立しない
        - 例えば null があるカラムで `where age=20 or age!=20 or age is not null` としないと全件取れない
    - **`not in` のサブクエリで使用されるテーブルの選択列に `null` が含まれると、SQL全体の結果は常に空になる**
        - 一つの例として、not in .... と not exists が同値ではない
            - in より exists のほうがパフォーマンスが良い？ （これは同値なので書き換え可能）
            - しかし not in, not exists が同じ結果にならない

    - 限定述語(`any`, `all`) と `null`
        - `any` は `in` と同値
        - `all` は 比較述語と併用して「〜全てと等しい」「〜全てよりも大きい」を表す
        - where age < all (select ...) としてサブクエリに null が含まれていると、全体として空の状態になる(すべてunkwnonになる)
        - よってデータから`null`を除くか`where age is not null` などと条件で除く必要があり

    - 限定述語と極値関数が同値ではなくなる
        - `where age < all(select age ...)` の代わりに `where age < (select min(age)... )` とすることもできる
        - この場合は `null` を除外しなくとも正しい結果になる
        - 極値集計の際に `null` は計算から除外されるため
        - 注意点
            - `where age < all(select age ...)` と `where age < (select min(age)... )` は、サブクエリの結果が空集合だったときの挙動が違う
            - all述語はデータをすべて返す、極値関数は null を返す → 比較対象がないときに全行を返すのと1行も返さないのとではどちらが良いかは要件による
    
    - 集約関数と`null`
        - サブクエリが空集合になるときには、`count` 以外の集計関数も `null` を返す

In [ ]:
%%sql

drop table if exists class_a;
create table class_a (
    name varchar(100),
    age integer,
    city varchar(100)
);
insert into class_a (name, age, city) values
('ブラウン', 22, '東京'),
('ラリー', 19, '埼玉'),
('ボギー', 21, '千葉');

drop table if exists class_b;
create table class_b (
    name varchar(100),
    age integer,
    city varchar(100)
);
insert into class_b (name, age, city) values
('斎藤', 22, '東京'),
('田尻', 23, '東京'),
('山田', null, '東京'),
('和泉', 18, '千葉'),
('武田', 20, '千葉'),
('石川', 19, '神奈川');

In [ ]:
%%sql
-- class_b の東京在住の生徒と年齢が一致しない class_a の生徒を抽出
-- 以下は一人も抽出されない
-- サブクエリが (22, 23, null) になり、結果的に where (age!=22) and (age!=23) and (age!=null) を評価することになり、
-- **AND の評価分の中に null が含まれると全体として `unknown` になる** ので常に結果は空になる
select * from class_a
where age not in (
    select age from class_b
    where city = '東京'
);



In [ ]:
%%sql

-- 正しくは exists 述語を使用する
select * from class_a
where not exists (
    select * from class_b
    where 
        class_a.age = class_b.age
        and class_b.city = '東京'
)



where exists (select ...) における挙動
1. class_a の1行目を取り出す `('ブラウン', 22, '東京')`
2. その行の値 age を使って class_b を検索する (この機能が相関サブクエリ)
3. exists / not exists の条件を判定する
4. 次の行へいく


`where 22 = class_b.age and class_b.city = '東京'`
- これを class_b の各行に適応して
    - 一致するものがあれば true, 
    - 最後まで検索して存在しなかったら false 
- を返す (not exists は反対の結果を返す) 


In [ ]:
%%sql
--all について
-- class_b　の東京在住の誰よりも若い、class_a の生徒
select *
from class_a
where age < all (
    select age from class_b
    where 
        city = '東京'
        and age is not null
    -- null が age に入ると全ての結果が unknown になってしまうので、データから除くか where で除く必要あり
)

In [ ]:
%%sql
-- ただし普通にかくならこっちがわかりやすい
select *
from class_a
where age < (
    select min(age) from class_b
    where 
        city = '東京'
        -- and age is not null
        -- 極値関数を使うとnullを除かなくとも機能する
)

In [ ]:
%%sql
-- count 以外の集計関数は、サブクエリが空集合だと null を返す
select * from class_a
where age < (
    select avg(age)
    from class_b
    where city = '埼玉' -- 空集合になる
)

In [ ]:
%%sql
select * from class_b

In [ ]:
%%sql
-- null が含まれる列のソート ACS
select * from class_b
order by age;

In [ ]:
%%sql
-- null が含まれる列のソート DESC
-- null が最初に来ちゃう
select * from class_b
order by age desc

In [ ]:
%%sql
-- null との文字列結合
-- coasesce の使い方
SELECT 
    null || 'abc' as null_str,
    '' || 'abc' as str0_str
    -- nullif(a, b)　は
    -- case when a = b then null else a end 
    -- と同じ挙動



    

- `nullif()` について
    - `nullif(a, b)` は 以下と同じ挙動
    ```sql
        case
            when a = b then null
            else a
        end 
    ```
    - 使用方法
    ```sql
    -- 0 除算を避ける
    select
        amount / nullif(amount_total, 0)
        -- amount_total が 0 のときは null になる


# EXISTS 述語の使い方

## 述語論理とは
- RDB, SQL ににおいて2つの基礎理論がある
    - 集合論
    - 述語論理（一階述語論理）
- 述語(predicate)とはなにか
    - 戻り値が真理値になる関数のこと
        - =, <, >, between, like, in, is null etc...
    - 引数の違いによる分類 (階: order)
        - 一階述語: =, <, >, between のようにテーブルの1行を入力とする
        - 二階述語: exists, not exists のように行の集合を入力とする

## Existsについて
- =, <>, between などは単一の値、スカラ値を引数に取るが、existsはサブクエリを引数に取る(=行の集合が引数)
- サブクエリのselect 句は何でも良い `select *`, `select 'xxxx'`, `select col`
- そして=, <>. between などは `true`, `false`, `unknown` の3値を返すのに対し、EXISTSは `true`, `false` の2パターンのみ返す
- 全称量化子と存在量化子
    - $\exists x\, P(x)$ (条件Pを満たすxが存在する) を表したのが exists 構文
    ```sql
    where exists (
        select *
        from t
        where (条件P)
    )
    ```
    - $\forall x\, P(x)$ を表した構文は存在しないが、以下のようにド・モルガンの法則により変換ができる
    - $\forall x\, P(x) \equiv \neg \exists x\, \neg p(x)$
        - 条件Pを満たすxが存在する = すべてのxが条件Pを満たさないわけではない
    - これを sql で表すと
    ```sql
    where not exists (
        select *
        from t
        where not (条件P)
    )
    ```

In [ ]:
%%sql
drop table if exists meetings;
create table meetings (
    meeting varchar(50),
    person varchar(50)
);
insert into meetings (meeting, person) values
('第1回', '伊藤'),
('第1回', '水島'),
('第1回', '坂東'),
('第2回', '伊藤'),
('第2回', '宮田'),
('第3回', '坂東'),
('第3回', '水島'),
('第3回', '宮田');

select * from meetings;

In [ ]:
%%sql
-- ここから、どれか少なくとも1回は参加したけど、他の会を欠席した人のリストを出したい
-- 全員が参加した場合の集合は、meeting と person 全組み合わせを得ること -> cross join
select distinct
    m1.meeting,
    m2.person
from meetings as m1
cross join meetings as m2

In [ ]:
%%sql
-- ちなみに2つのカラムの組み合わせの全パターンを求めるには、select distinct する必要あり
with
tmp (col_a, col_b, col_c) as (
    values
        ('A', 'a', 1),
        ('A', 'b', 2),
        ('B', 'c', 1)
)

select distinct
    t1.col_a,
    t2.col_b,
    t3.col_c
from tmp as t1
cross join tmp as t2
cross join tmp as t3

In [ ]:
%%sql
-- この12個のうちで、もとのテーブルに存在しないデータだけ抽出する
select distinct
    m1.meeting,
    m2.person
from meetings as m1
cross join meetings as m2
-- 上部で直積を計算、その後 where not exists で存在しない条件を付加
where not exists (
    select *
    from meetings as m3
    where 
        m1.meeting = m3.meeting
        and m2.person = m3.person
)


### 全称量化 (すべての行について〜)の変換

すべての行について〜という全称量化の表現を、「〜でない行が一つも存在しない」という二重否定文へ変換する

In [ ]:
%%sql
drop table if exists test_scores;
create table test_scores (
    student_id integer,
    subject varchar(100),
    score integer
);
insert into test_scores (student_id, subject, score) values
(100, '算数', 100),
(100, '国語', 80),
(100, '理科', 80),
(200, '算数', 80),
(200, '国語', 95),
(300, '算数', 40),
(300, '国語', 90),
(300, '社会', 55),
(400, '算数', 80);
select * from test_scores

In [ ]:
%%sql
-- すべての教科について50点以上を取っている生徒を選択する
-- つまり：50点未満である教科が一つも存在しない

select distinct
    student_id
from test_scores as t1
where not exists (
    select *
    from test_scores as t2
    where 
        t2.student_id = t1.student_id
        and t2.score < 50
)


In [ ]:
%%sql
-- こうもかけるのでは？
select
    student_id
from test_scores
group by student_id
having min(score) >= 50



## Having 

- 